# スマホ用の軽いボール検出モデルを作る（学習）

**使い方**: 「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選び、「ランタイム」→「**すべてのセルを実行**」。
途中で Google ドライブへのアクセス許可を求められたら許可してください。終わると `マイドライブ/テニス解析/ball_small.onnx` ができます。
解析画面（https://air-tomo.github.io/cut/analyze/）で、このファイルを「ボール検出モデル」として選ぶと使えます（一度選べば端末に保存されます）。

- 学習データ: TrackNet のテニスデータ（放送映像 10 試合・約 2 万フレーム）と、自分の試合動画（IMG_0131、ボール位置は TrackNet で求めたもの）
- 1〜2 時間ほどかかります。途中で止まった場合は、もう一度「すべてのセルを実行」してください

In [ ]:
!nvidia-smi -L || echo "GPU がありません（ランタイムのタイプで T4 GPU を選んでください）"
!pip -q install gdown onnx
!rm -rf repo && git clone -q --depth 1 https://github.com/air-tomo/air-tomo.github.io repo
%cd repo/golf-swing-app/tools/court3d

In [ ]:
# TrackNet のテニスデータ（約 2.5GB）
import os
if not os.path.exists('/content/Dataset'):
    !gdown -q "https://drive.google.com/uc?id=1DQ3ZbvokTsgOq6x-ay6O8U2W4a8e3LFw" -O /content/Dataset.zip
    !unzip -q /content/Dataset.zip -d /content && rm /content/Dataset.zip
!ls /content/Dataset | head

In [ ]:
# 自分の試合動画から学習データを作る（ドライブの IMG_0131.MOV）
import glob, shutil
from google.colab import drive
drive.mount('/content/drive')
cands = glob.glob('/content/drive/MyDrive/**/IMG_0131.MOV', recursive=True)
if cands:
    if not os.path.exists('/content/IMG_0131.MOV'): shutil.copy(cands[0], '/content/IMG_0131.MOV')
    !python3 make_extra.py /content/IMG_0131.MOV data/IMG_0131_tn /content/extra | tail -3
else:
    print('IMG_0131.MOV が見つからないので、TrackNet のデータだけで学習します')

In [ ]:
# 学習（T4 で 1〜2 時間）
extra = '--extra /content/extra' if os.path.exists('/content/extra') else ''
!python3 train_small.py --tracknet /content/Dataset {extra} --epochs 25 --batch 16 --out /content/ball_small

In [ ]:
# ドライブに保存
os.makedirs('/content/drive/MyDrive/テニス解析', exist_ok=True)
shutil.copy('/content/ball_small.onnx', '/content/drive/MyDrive/テニス解析/ball_small.onnx')
print('保存しました: マイドライブ/テニス解析/ball_small.onnx')